In [45]:
import sys
import os
# Get the current directory and parent directory
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)
sys.path.insert(0, current_dir)

# Import from the-machine module
import importlib.util
the_machine_path = os.path.join(parent_dir, "the-machine")
sys.path.insert(0, the_machine_path)

# Import all from the-machine and custom_matroid_functions
from HM_module import *
from custom_matroid_functions import *

def whitney_numbers2(M):
    r"""
    Return the Whitney numbers of the second kind of the matroid.

    The Whitney numbers of the second kind are here encoded as a vector
    (W_0, ..., W_r), where W_i is the number of flats of rank i,
    and r is the rank of the matroid.

    INPUT:
    - M -- a matroid

    OUTPUT: list of integers

    EXAMPLES::

        sage: M = matroids.catalog.BetsyRoss()
        sage: whitney_numbers2(M)
        [1, 11, 20, 1]
    """
    L = M.lattice_of_flats()
    r = M.rank()
    W = [ZZ.zero()] * (r + 1)
    for F in L:
        W[M.rank(F)] += 1
    return W

In [46]:
P = LaurentPolynomialRing(ZZ, 'x')
x = P.gen()
k, n = 5, 5

M = matroids.CompleteGraphic(n)
#M = matroids.Uniform(k, n)
#M = matroids.catalog.Fano()
print(whitney_numbers2(M))
E = M.groundset()
L = M.lattice_of_flats()
flats_set = set(L)
r = M.rank()
rank = lambda x : M.rank(x)
mobius = lambda M: characteristic_polynomial(M)(0)
chow = chow_polynomial
redchar = lambda M: reduced_characteristic_polynomial(M)(x)
char_poly = lambda M: characteristic_polynomial(M)(x)
kl_poly = lambda M: M.lattice_of_flats().kazhdan_lusztig_polynomial()(x)
invkl = lambda M: (-1)**(M.rank()) * inverse_kazhdan_lustig_polynomial(M)(x)
klpoly_rev = lambda M: kl_poly(M)(x**-1) * x**M.rank()

polys = [chow, redchar, char_poly, kl_poly, invkl, klpoly_rev]

print(chow_polynomial(M))
print(reduced_characteristic_polynomial(M))
print(char_poly(M))
print(kl_poly(M))
print(invkl(M))
print(mobius(M))
print(klpoly_rev(M))

[1, 10, 25, 15, 1]
x^3 + 41*x^2 + 41*x + 1
x^3 - 9*x^2 + 26*x - 24
24 - 50*x + 35*x^2 - 10*x^3 + x^4
1 + 5*x
24 + 10*x
24
5*x^3 + x^4


In [47]:
def convolve(f, g):
    flats = list(flats_set)
    sum = P(0)
    for F in flats:
        sum += f(M.delete(E - F))(x) * g(M.contract(F))(x)
    return sum


In [48]:
print(convolve(chow, klpoly_rev))
print(convolve(chow, redchar))
print(convolve(redchar, chow))
print(convolve(chow, char_poly))
print(convolve(char_poly, chow))
print(convolve(chow, invkl))
print(convolve(invkl, chow))
print(convolve(chow, kl_poly))
print(convolve(kl_poly, chow))


1 + 56*x + 166*x^2 + 56*x^3 + x^4
0
0
x + 41*x^2 + 41*x^3 + x^4
x + 41*x^2 + 41*x^3 + x^4
x + 26*x^2 + x^3
x + 31*x^2 + x^3
52 + 171*x + 56*x^2 + x^3
52 + 156*x + 51*x^2 + x^3


In [49]:

print(convolve(kl_poly, redchar))
print(convolve(redchar, kl_poly))
print(convolve(kl_poly, char_poly))
print(convolve(char_poly, kl_poly))
print(convolve(kl_poly, chow))
print(convolve(chow, kl_poly))
print(convolve(kl_poly, invkl))
print(convolve(invkl, kl_poly))

x + x^2 + x^3
x + 6*x^2 + x^3
5*x^2 + x^4
5*x^3 + x^4
52 + 156*x + 51*x^2 + x^3
52 + 171*x + 56*x^2 + x^3
0
0


In [50]:
print(convolve(char_poly, invkl))
print(convolve(invkl, char_poly))

228 - 315*x + 145*x^2 - 25*x^3 + x^4
228 - 300*x + 125*x^2 - 20*x^3 + x^4


In [51]:
print(convolve(redchar, invkl))
print(convolve(invkl, redchar))
print(convolve(redchar, char_poly))
print(convolve(char_poly, redchar))

-228 + 111*x - 24*x^2 + x^3
-228 + 96*x - 19*x^2 + x^3
-228 + 446*x - 284*x^2 + 61*x^3 - x^4
-228 + 446*x - 284*x^2 + 61*x^3 - x^4


In [52]:
def augment1(M, f):
    sum = P(0)
    flats = list(M.lattice_of_flats())
    #flats.remove(frozenset())
    for F in flats:
        sum += x**rank(F) * f(M.contract(F))(x)
    return sum

print("augmented chow", augment1(M, chow_polynomial))
e = next(x for x in E if x not in M.coloops())
N = M.delete(e)
print("augmented del", augment1(N, chow_polynomial))

augmented chow 1 + 51*x + 146*x^2 + 51*x^3 + x^4
augmented del 1 + 46*x + 128*x^2 + 46*x^3 + x^4
